# Settings

In [ ]:
import mne
import numpy as py
import pandas as pd
import os
import sys
import numpy as np
import matplotlib
import pickle
from mne import read_labels_from_annot
import scipy
from mne.beamformer import make_lcmv, apply_lcmv
from pathlib import Path



matplotlib.use('Qt5Agg')
mne.set_log_level(verbose=30)
import matplotlib.pyplot as plt

In [ ]:
PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR != PROJECT_DIR.parent and not (PROJECT_DIR / "project_config.json").is_file():
    PROJECT_DIR = PROJECT_DIR.parent
if not (PROJECT_DIR / "project_config.json").is_file():
    raise FileNotFoundError("Could not find project_config.json above the notebook working directory")
sys.path.insert(0, str(PROJECT_DIR))
from project_config import get_path

PATH = get_path("project_root")
expt = 'IdvDiff'
meg_dir = get_path("meg")
mri_dir = get_path("mri")
log_dir = get_path("logs")
fig_dir = str(get_path("figures"))
stc_homedir = str(get_path("stc"))

# get subject names as a list of directories
subjects =  [f for f in os.listdir(meg_dir) if 'R'in f and '.md' not in f]

n_subj = len(subjects)
# Create dictionaries to store subject variables
subj_evokeds = {}
infos = {}

save_plot = True # settings for the evoked plots
overwrite = True # do not overwrite existing covariance etc
crop = False

os.chdir(PATH)
from helperfunctions import select_condition, get_source_reqs, beamformer_prereqs


In [ ]:
mapping = {
    'scrambled_fast': 'WordList',
    'scrambled_slow': 'WordList',
    'sents_fast': 'SentList',
    'sents_slow': 'SentList',
    'intact_fast': 'Story',
    'intact_slow': 'Story'
}

# define conditions
conditions = ['scrambled_fast', 'scrambled_slow',
        'sents_fast', 'sents_slow',
        'intact_fast', 'intact_slow'] 

conditions_simple = set([s.split('_')[0] for s in conditions])

# Create evoked

In [ ]:
# adapted from a larger subject loop
for subj in subjects:
    # Create all nec directories
    # figure dir
    if not os.path.exists(os.path.join(fig_dir,subj)):
        os.makedirs(os.path.join(fig_dir,subj))
    # source dir
    if not os.path.exists(os.path.join(meg_dir,subj,'source')):
        os.makedirs(os.path.join(meg_dir,subj,'source'))
    # whitened evoked dir
    if not os.path.exists(os.path.join(fig_dir,subj,'whitened')):
        os.makedirs(os.path.join(fig_dir,subj,'whitened'))

    print(f'Creating evoked responses for {subj}')
    cleandata = mne.read_epochs(os.path.join(meg_dir, subj, 'preproc', f'{subj}_{expt}_final-epo.fif'))
                
    # add info object to use later
    info = cleandata.info
    infos[subj]= info
    print('Data read in...')
    # select and define all of the conditions 
    trials = cleandata["Type=='Trial'"] # filter out the personal experience and task segments

    evoked = {}
    for condition in conditions:
        # crop the fast trials down
        print(condition)
        data = select_condition(trials,condition)
        cond = data.average() # average over conditions
        cond.comment = condition
        if save_plot:
            cond_plot = cond.plot_joint(title= cond.comment, show=False, times='auto');
            cond_plot.savefig(f'figures/{subj}/{subj}_evoked-{cond.comment}.png')
            plt.close()
        evoked[condition]=cond

    # add to final dict
    subj_evokeds[subj]= evoked      
    print('Full condition list finished...\n')

In [ ]:
subj_evokeds

### Sanity check: plot all evoked for one subj

In [ ]:
for subj in subjects:
    all_evokeds = mne.combine_evoked(list(subj_evokeds[subj].values()), weights='equal')
    evoked_plot = all_evokeds.plot_joint(show=False, ts_args=dict(window_title=f'{subj}'))
    evoked_plot.comment = 'grand_avg'
    evoked_plot.savefig(f'figures/{subj}/{subj}_allevoked.png')
plt.show()

# Create Source Timecourses

## Covariance and whitened data

In [ ]:
# compute the noise covariance and plot whitened data as a sanity check; make a function as well
for subj in subjects:
    print(f'Starting subject {subj}')
    cov_fname = os.path.join(meg_dir, subj,f'source/{subj}-cov.fif')
    if not os.path.isfile(cov_fname):
      print('Getting raw file first ...')
      raws = [mne.io.read_raw_fif(os.path.join(meg_dir,subj, 'preproc',i), preload=True) for i in os.listdir(os.path.join(meg_dir,subj,'preproc')) if i.endswith('raw.fif')]
      rawfile =  raws[0]
      # load the excluded channels 
      print("Loading bad channels!")
      cache_dir = os.path.join(meg_dir, subj,'cache')
      with open(cache_dir + f'/bad_channels.p', 'rb') as file: 
        bads = pickle.load(file)
        print(bads)
      rawfile.info['bads'] = bads
      # calculate covariance DO NOT noise reduce emptyroom
      cov = mne.compute_raw_covariance(rawfile, method=['shrunk', 'diagonal_fixed', 'empirical'], tmin=0, tmax=120,n_jobs=-1, picks=['meg']) # first 2 min are emptyroom
      cov.save(cov_fname, overwrite=overwrite)
      print('Covariance calculated! (and saved)')
    
    print('Covariance already exists... Updating whitened plots')
    cov = mne.read_cov(cov_fname)

    evoked = subj_evokeds[subj].values()

    for ev in evoked:
      fig = ev.plot_white(noise_cov=cov, show=False)
      fig.savefig(f'figures/{subj}/whitened/whitened-{ev.comment}.png')
    print('Full condition list finished...\n')

In [ ]:
subj_evokeds[subj].values()

## Create source space

## Create stcs

### Fsaverage sym

In [ ]:
for subj in subjects:
    print(f'Creating morphed stcs for subject {subj}')
    # load info object
    info = infos[subj]
    # get evoked for subject
    evoked = subj_evokeds[subj].values()   
    # calculate inverse solution
    inv = get_source_reqs(subj, info)
    SNR = 3 # 3 for evoked, 2 for single trial analysis
    lambda2 = 1.0 / SNR ** 2.0 # regularization parameter
    # generate and morph stcs
    for ev in evoked:
        stc = mne.minimum_norm.apply_inverse(ev, inv, lambda2=lambda2, method='dSPM')
        stc.subject = subj
        # morphing stcs to the fsaverage using precomputed matrix method:
        morph = mne.compute_source_morph(stc, subject_from=subj, subject_to='fsaverage_sym', subjects_dir=mri_dir, spacing = 4)
        stc_dir = os.path.join(stc_homedir, ev.comment)
        stc_fname = f'{subj}_{ev.comment}-sym_dSPM'
        print(f'Morphing {stc_fname} to fsaverage-sym...')
        stc_fsaverage = morph.apply(stc)
        if not os.path.exists(stc_dir):
            os.makedirs(stc_dir)
        stc_fsaverage.save(os.path.join(stc_dir, stc_fname), overwrite=True)

    print(f'Finished subject {subj} \n')

## Create beamformers

In [ ]:
epoch_tmin = 0
epoch_tmax = 0.45

for subj in subjects:
    cleandata = mne.read_epochs(os.path.join(meg_dir, subj, 'preproc', f'{subj}_{expt}_final-epo.fif'))
    filters = beamformer_prereqs(subj, cleandata, meg_dir)
    evoked = subj_evokeds[subj].values()   
    for ev in evoked:
        stc = apply_lcmv(ev, filters)
        stc.subject = subj
        #stc.plot(subject=subj, subjects_dir=mri_dir,hemi='split')        
        # morphing stcs to the fsaverage using precomputed matrix method:
        morph = mne.compute_source_morph(stc, subject_from=subj, subject_to='fsaverage', subjects_dir=mri_dir, spacing = 4)
        stc_dir = os.path.join(stc_homedir, ev.comment)
        stc_fname = f'{subj}_{ev.comment}_beamformer'
        print(f'Morphing {stc_fname} to fsaverage...')
        stc_fsaverage = morph.apply(stc)
        if not os.path.exists(stc_dir):
            os.makedirs(stc_dir)
        stc_fsaverage.save(os.path.join(stc_dir, stc_fname), overwrite=True)


# Easy plotting for presentation

## Plotting evoked responses

In [ ]:
# easy evoked plotting
cond_fast = [cond for cond in conditions if 'fast' in cond]
cond_slow = [cond for cond in conditions if 'slow' in cond]
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=plt.cm.copper(np.linspace(0, 1,3))) # have plot cycle through colors

fig, axs = plt.subplots(nrows=2, ncols=1,figsize=(12, 8), sharey=True, squeeze=False)
fig.subplots_adjust(wspace=0.1,hspace=0.5)
    
for row, cond_group in enumerate([cond_fast, cond_slow]):            
    for condition in cond_group:
        subj_avg = []
        for subj in subjects:
            ev = subj_evokeds[subj][condition] # read in evoked
            sem = scipy.stats.sem(ev.data)
            subj_avg.append(ev.data.mean(0)) # average across channels

        avg_arr = np.mean(np.array(subj_avg), axis=0)  # Average over subjects
        axs[row,0].plot(ev.times*1000, avg_arr, label=mapping[condition])

        # Formatting subplots
        if 'fast' in condition:
            axs[row,0].set_xlim(-100,450)
        else:
            axs[row,0].set_xlim(-100,800)
        axs[row,0].axvline(x=0, color='k', linestyle='--')
        axs[row,0].set_title(f'Evoked comparison ({["Fast", "Slow"][row]}) n={n_subj}', fontsize=14)
        axs[row,0].set_ylabel('Mean evoked value')
        axs[row,0].set_xlabel('Time [ms]')
        axs[row,0].legend()
    
print(f'Plot finished!')
fig.suptitle(f'Evoked comparisons', fontsize=18)
fig.tight_layout()
#fig.savefig(fig_dir+f'/Evoked Comparisons-n={n_subj}')
plt.show()


## Plotting STCS

In [ ]:
# read stcs, average activity within a given region, plot average on grid
os.environ["SUBJECTS_DIR"] = str(mri_dir)

hemis = ['lh', 'rh']
labels = read_labels_from_annot(subject='fsaverage', parc='PALS_B12_Brodmann', subjects_dir=mri_dir)
ROIs = ['17', '20', '21', '22', '38', '39', '11', '44', '45', '47']
cond_fast = [cond for cond in conditions if 'fast' in cond]
cond_slow = [cond for cond in conditions if 'slow' in cond]
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=plt.cm.copper(np.linspace(0, 1,3))) # have plot cycle through colors


for BA in ROIs:
    fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(12, 8), sharey=True, squeeze=False)
    fig.subplots_adjust(wspace=0.1,hspace=0.5)
    
    for col, hemi in enumerate(hemis):
        for row, cond_group in enumerate([cond_fast, cond_slow]):            
            for condition in cond_group:
                subj_avg = []
                for subj in subjects:
                    filename = os.path.join(stc_homedir, 'signed',condition, f'{subj}_{condition}-sym_dspm-{hemi}.stc') # signed
                    try:
                        cond_stc = mne.read_source_estimate(filename)
                        #print(f"loading in {subj}'s data...")
                        try:
                            current_label = next(label for label in labels if label.name == f'Brodmann.{BA}-{hemi}')
                            #current_label_morph = current_label.morph(subject_to='fsaverage_sym', grade = 4)
                            label_avg = cond_stc.in_label(current_label)                            
                            subj_avg.append(label_avg.data.mean(0))
                        except StopIteration:
                            print(f"Brodmann {BA}-{hemi} does not exist")
                            continue
                    except OSError:
                        print(f"No file found: {filename}")
                        continue

                avg_arr = np.mean(np.array(subj_avg), axis=0)  # Average over subjects
                axs[row,col].plot(cond_stc.times*1000, avg_arr, label=mapping[condition])

                # Formatting subplots
                if 'fast' in condition:
                    axs[row,col].set_xlim(-100,450)
                else:
                    axs[row,col].set_xlim(-100,800)
                axs[row,col].axvline(x=0, color='k', linestyle='--')
                axs[row,col].set_title(f'BA {BA} - {hemi} ({["Fast", "Slow"][row]})', fontsize=14)
                axs[row,col].set_ylabel('Mean dSPM Value')
                axs[row,col].set_xlabel('Time [ms]')
                axs[row,col].legend()
    
    print(f'Plot finished for BA {BA}')
    fig.suptitle(f'Comparisons for Brodmann Area {BA}-sym', fontsize=18)
    fig.tight_layout()
    fig.savefig(fig_dir+f'/Comparisons for Brodmann Area {BA}-n={n_subj}-signed')

plt.show()